# 11 — Visualization in Workflow

**Purpose:** Prove every major `draw()` option runs without error and renders an
inline preview. This is a **functional smoke pass**, not a pixel-nit review
(the visual pack handles that). Every rendered image appears inline so JMT can
see thumbnails without opening PDFs.

**Surfaces covered:**
- [ ] `trace.draw()` — baseline call
- [ ] `vis_mode` — `'unrolled'` (default) and `'rolled'`
- [ ] `node_mode` — `'default'`, `'profiling'`, `'vision'`, `'attention'`
- [ ] `module` — module-focus subgraph
- [ ] `show_containers` — `False`, `'labels'`, `'cluster'`, `'collapsed'`, `'auto'`
- [ ] `code_panel` — source-context side panel
- [ ] `node_overlay` — dict score overlay
- [ ] `vis_theme` — `'dark'`
- [ ] `show_legend`
- [ ] `direction` — `'leftright'`
- [ ] `show_buffer_layers` — `'always'`
- [ ] `vis_call_depth` — depth cap
- [ ] `trace.draw_backward()` — backward graph
- [ ] `trace.draw_combined()` — forward + backward combined
- [ ] `vis_fileformat` — `'png'` (used for inline display), `'svg'`, `'pdf'`
- [ ] `order_siblings=` — sibling-ordering pass (§15)
- [ ] `tl.viz.heatmap`, `tl.viz.histogram`, `tl.viz.channel_grid`, `tl.viz.render_heatmap` — viz primitives (§16)

## 1. Environment setup

In [ ]:
import pathlib
import sys
import tempfile
import os

sys.path.insert(0, str(pathlib.Path.cwd()))

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
import torchlens as tl
from _models import ZOO
from IPython.display import Image, display

# Temp dir for all rendered files (deleted with Python session)
TMPDIR = tempfile.mkdtemp(prefix="tl_viz_audit_")


def render_png(trace, stem, **kwargs):
    """Render trace to PNG and return the path."""
    outpath = os.path.join(TMPDIR, stem)
    trace.draw(vis_outpath=outpath, vis_save_only=True, vis_fileformat="png", **kwargs)
    return outpath + ".png"


def show(path, caption=""):
    """Confirm a render succeeded (text only).

    Graph images deliberately live in the visual pack (visual/visual_audit.pdf), not
    embedded here: this notebook verifies every draw option *runs*, and embedding ~28
    PNGs bloats the .ipynb to multiple MB and trips the repo secret/large-file hooks.
    """
    size_kb = os.path.getsize(path) // 1024
    label = caption or os.path.basename(path)
    print(f"  {label}  [{size_kb} KB]  -> rendered OK")


print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")
print(f"Temp dir          : {TMPDIR}")

## 2. Baseline — `trace.draw()` defaults (unrolled, tiny_mlp)

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_mlp = tl.trace(model, x)

path = render_png(trace_mlp, "00_baseline")
show(path, "tiny_mlp — default (unrolled, default node_mode)")

## 3. `vis_mode` — `'unrolled'` vs `'rolled'`

`reused_relu_loop` calls a single `nn.ReLU` module 4x — good test for rolling.

In [ ]:
model, x = ZOO["reused_relu_loop"]()
trace_loop = tl.trace(model, x)

path_unrolled = render_png(trace_loop, "01_unrolled", vis_mode="unrolled")
show(path_unrolled, "reused_relu_loop — vis_mode='unrolled'")

path_rolled = render_png(trace_loop, "02_rolled", vis_mode="rolled")
show(path_rolled, "reused_relu_loop — vis_mode='rolled' (self-loop collapsed)")

## 4. `node_mode` — `'default'`, `'profiling'`, `'vision'`, `'attention'`

In [ ]:
# tiny_branch_cnn has conv layers — good for vision mode
model, x = ZOO["tiny_branch_cnn"]()
trace_cnn = tl.trace(model, x)

for mode in ["default", "profiling", "vision", "attention"]:
    try:
        path = render_png(trace_cnn, f"03_node_{mode}", node_mode=mode)
        show(path, f"tiny_branch_cnn — node_mode='{mode}'")
    except Exception as exc:
        print(f"⚠️ node_mode='{mode}' ERROR: {exc}")

## 5. `module` — subgraph focus

Pass a module address string to zoom the graph to ops inside that module.

In [ ]:
model, x = ZOO["demo_model"]()
trace_demo = tl.trace(model, x)

print(
    "Available module addresses:",
    list(trace_demo.modules.keys() if hasattr(trace_demo.modules, "keys") else []),
)
print()

path_full = render_png(trace_demo, "04_demo_full")
show(path_full, "demo_model — full graph")

path_focus = render_png(trace_demo, "05_demo_focus", module="inner_module")
show(path_focus, "demo_model — module='inner_module' (focus subgraph)")

## 6. `show_containers` — `False`, `'labels'`, `'cluster'`, `'collapsed'`, `'auto'`

`mid_graph_container` returns a dict mid-graph, useful to exercise all container modes.

In [ ]:
model, x = ZOO["mid_graph_container"]()
trace_cont = tl.trace(model, x)

for mode in [False, "labels", "cluster", "collapsed", "auto"]:
    stem = f"06_cont_{str(mode).replace('/', '_')}"
    try:
        path = render_png(trace_cont, stem, show_containers=mode)
        show(path, f"mid_graph_container — show_containers={mode!r}")
    except Exception as exc:
        print(f"⚠️ show_containers={mode!r} ERROR: {exc}")

## 7. `code_panel` — source-context side panel

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_mlp2 = tl.trace(model, x)

try:
    path = render_png(trace_mlp2, "07_code_panel", code_panel=True)
    show(path, "tiny_mlp — code_panel=True")
except Exception as exc:
    print(f"⚠️ code_panel ERROR: {exc}")

## 8. `node_overlay` — score dict overlay

`node_overlay` accepts a `dict[label, float]` or a built-in overlay name
(e.g. `'bytes'`, `'flops'`, `'magnitude'`).  Colours nodes by score.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_mlp3 = tl.trace(model, x)

# Custom dict overlay (importance scores)
importance = {
    "relu_1_2": 0.9,
    "linear_1_1": 0.4,
    "linear_2_3": 0.2,
}
try:
    path = render_png(trace_mlp3, "08_overlay_dict", node_overlay=importance)
    show(path, "tiny_mlp — node_overlay=dict (custom importance scores)")
except Exception as exc:
    print(f"⚠️ node_overlay dict ERROR: {exc}")

# Built-in 'bytes' overlay
try:
    path2 = render_png(trace_mlp3, "08_overlay_bytes", node_overlay="bytes")
    show(path2, "tiny_mlp — node_overlay='bytes' (tensor size)")
except Exception as exc:
    print(f"⚠️ node_overlay 'bytes' ERROR: {exc}")

## 9. `vis_theme` and `show_legend`

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_mlp4 = tl.trace(model, x)

try:
    path = render_png(trace_mlp4, "09_dark", vis_theme="dark")
    show(path, "tiny_mlp — vis_theme='dark'")
except Exception as exc:
    print(f"⚠️ vis_theme='dark' ERROR: {exc}")

try:
    path2 = render_png(trace_mlp4, "09_legend", show_legend=True)
    show(path2, "tiny_mlp — show_legend=True")
except Exception as exc:
    print(f"⚠️ show_legend ERROR: {exc}")

## 10. `direction` — layout orientation

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_mlp5 = tl.trace(model, x)

for d in ["bottomup", "topdown", "leftright"]:
    try:
        path = render_png(trace_mlp5, f"10_dir_{d}", direction=d)
        show(path, f"tiny_mlp — direction='{d}'")
    except Exception as exc:
        print(f"⚠️ direction='{d}' ERROR: {exc}")

## 11. `show_buffer_layers` and `vis_call_depth`

In [ ]:
# demo_model has a buffer (example_buffer)
model, x = ZOO["demo_model"]()
trace_demo2 = tl.trace(model, x)

for buf_mode in ["never", "meaningful", "always"]:
    try:
        path = render_png(trace_demo2, f"11_buf_{buf_mode}", show_buffer_layers=buf_mode)
        show(path, f"demo_model — show_buffer_layers='{buf_mode}'")
    except Exception as exc:
        print(f"⚠️ show_buffer_layers='{buf_mode}' ERROR: {exc}")

# vis_call_depth cap
try:
    path2 = render_png(trace_demo2, "11_depth1", vis_call_depth=1)
    show(path2, "demo_model — vis_call_depth=1 (top-level only)")
except Exception as exc:
    print(f"⚠️ vis_call_depth=1 ERROR: {exc}")

## 12. `draw_backward()` — backward graph

Requires calling `.backward()` on a scalar output before drawing.

In [ ]:
from _models import LinearReluModel

model_bwd = LinearReluModel()
x_bwd = torch.randn(1, 3, requires_grad=True)
trace_bwd = tl.trace(model_bwd, x_bwd)

# Run backward on scalar output
trace_bwd[trace_bwd.layer_labels[-1]].out.backward()

try:
    out_bwd = os.path.join(TMPDIR, "12_backward")
    trace_bwd.draw_backward(vis_outpath=out_bwd, vis_save_only=True, vis_fileformat="png")
    show(out_bwd + ".png", "LinearReluModel — draw_backward()")
except Exception as exc:
    print(f"⚠️ draw_backward ERROR: {exc}")

## 13. `draw_combined()` — forward + backward in one view

In [ ]:
try:
    out_comb = os.path.join(TMPDIR, "13_combined")
    trace_bwd.draw_combined(vis_outpath=out_comb, vis_save_only=True, vis_fileformat="png")
    show(out_comb + ".png", "LinearReluModel — draw_combined() (fwd + bwd)")
except Exception as exc:
    print(f"⚠️ draw_combined ERROR: {exc}")

## 14. `vis_fileformat` — PDF and SVG

PNG is used for inline display above; confirm PDF and SVG also write without error.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_fmt = tl.trace(model, x)

for fmt in ["pdf", "svg"]:
    try:
        outpath = os.path.join(TMPDIR, f"14_fmt_{fmt}")
        trace_fmt.draw(vis_outpath=outpath, vis_save_only=True, vis_fileformat=fmt)
        written = outpath + f".{fmt}"
        size_kb = os.path.getsize(written) // 1024
        print(f"vis_fileformat='{fmt}': OK ({size_kb} KB written to {written})")
    except Exception as exc:
        print(f"⚠️ vis_fileformat='{fmt}' ERROR: {exc}")

## 15. `order_siblings=` draw kwarg

`order_siblings=True` (the default) applies a sibling-ordering pass to the Graphviz dot
layout for forward-unrolled graphs.  Set to `False` to use the raw dot layout without
reordering.  Both variants are smoke-tested here; visual diff lives in the visual pack.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace_os = tl.trace(model, x)

for ordered in [True, False]:
    stem = f"15_order_siblings_{ordered}"
    outpath = os.path.join(TMPDIR, stem)
    try:
        trace_os.draw(
            vis_outpath=outpath, vis_save_only=True, vis_fileformat="pdf", order_siblings=ordered
        )
        size_kb = os.path.getsize(outpath + ".pdf") // 1024
        print(f"order_siblings={ordered!s:5s}: OK ({size_kb} KB written)")
    except Exception as exc:
        print(f"order_siblings={ordered!s:5s}: ERROR — {exc}")

print()
print("Note: order_siblings=True is the default (canonical CLAUDE.md recommended setting).")
print("Visual diff between True/False is only visible on graphs with branching siblings.")

## 16. `tl.viz` primitives — `heatmap`, `channel_grid`, `histogram`, `render_heatmap`

`tl.viz.heatmap`, `tl.viz.channel_grid`, and `tl.viz.histogram` are **factory functions**
that return a node-spec callback: `fn = tl.viz.heatmap()` then `img = fn(tensor)`.
`tl.viz.render_heatmap` is a direct renderer that returns a PIL Image immediately.

These are primarily designed as callbacks for `node_mode` node specs, but can also be called
standalone on any tensor.  Results are PIL `Image` objects; at most one small preview is
embedded here to keep the notebook under the size limit.

In [ ]:
import torch as _torch

model, x = ZOO["tiny_mlp"]()
trace_viz = tl.trace(model, x, save=tl.func("relu"))
relu_tensor = trace_viz["relu_1_2"].out.detach()

print("relu activation shape:", relu_tensor.shape)
print()

# 1. tl.viz.heatmap — factory -> callback -> PIL Image
heatmap_fn = tl.viz.heatmap(max_size=120)  # factory call
print("tl.viz.heatmap(max_size=120) type:", type(heatmap_fn).__name__)
try:
    hm_img = heatmap_fn(relu_tensor)
    if hm_img is not None:
        print(f"  heatmap_fn(relu_tensor) -> PIL Image, size={hm_img.size}")
    else:
        print("  heatmap_fn returned None (tensor too small for requested size)")
except Exception as exc:
    print(f"  heatmap_fn ERROR: {exc}")
print()

# 2. tl.viz.histogram — factory -> callback -> PIL Image
histogram_fn = tl.viz.histogram(bins=20, width=160, height=80)
print("tl.viz.histogram(bins=20) type:", type(histogram_fn).__name__)
try:
    hist_img = histogram_fn(relu_tensor)
    if hist_img is not None:
        print(f"  histogram_fn(relu_tensor) -> PIL Image, size={hist_img.size}")
    else:
        print("  histogram_fn returned None")
except Exception as exc:
    print(f"  histogram_fn ERROR: {exc}")
print()

# 3. tl.viz.channel_grid — factory -> callback -> PIL Image
channel_grid_fn = tl.viz.channel_grid(n=4, max_size=64)
print("tl.viz.channel_grid(n=4) type:", type(channel_grid_fn).__name__)
try:
    cg_img = channel_grid_fn(relu_tensor)
    if cg_img is not None:
        print(f"  channel_grid_fn(relu_tensor) -> PIL Image, size={cg_img.size}")
    else:
        print(
            "  channel_grid_fn returned None (tensor shape not compatible; expects 4D conv activations)"
        )
except Exception as exc:
    print(f"  channel_grid_fn ERROR: {exc}")
print()

# 4. tl.viz.render_heatmap — direct PIL renderer (not a factory)
try:
    rh_img = tl.viz.render_heatmap(relu_tensor, width=80, height=40)
    print(f"tl.viz.render_heatmap(relu, width=80, height=40) -> PIL Image, size={rh_img.size}")
    # Save a tiny preview (no inline embedding to keep notebook small)
    rh_path = os.path.join(TMPDIR, "16_render_heatmap.png")
    rh_img.save(rh_path)
    size_kb = os.path.getsize(rh_path) // 1024
    print(f"  saved preview to {rh_path} ({size_kb} KB)")
except Exception as exc:
    print(f"  render_heatmap ERROR: {exc}")

print()
print("Note: channel_grid is designed for 4D conv activations (N,C,H,W); returns None for")
print("      2D tensors. heatmap and histogram work on any tensor shape.")

## 15. Option run summary

In [ ]:
# List all PNGs written to verify coverage
pngs = sorted(f for f in os.listdir(TMPDIR) if f.endswith(".png"))
print(f"Total PNG renders: {len(pngs)}")
for f in pngs:
    size_kb = os.path.getsize(os.path.join(TMPDIR, f)) // 1024
    print(f"  {f:40s}  {size_kb:4d} KB")

---

## ⚠️ GAPs / ergonomic smells

- **`node_overlay` does not accept a callable** — `node_overlay=lambda op: ...` raises `TypeError: argument of type 'function' is not iterable`. Only `str` (built-in name) or `dict[label, float]` works. A callable would be far more natural for programmatic overlays; current API forces the user to pre-build a dict with exact label strings.
- **`vis_save_only=True` is required to suppress auto-opening** — without it, draw() tries to open the PDF in the system viewer, which hangs in headless/notebook environments.  The flag name (`vis_save_only`) is somewhat unexpected; `display=False` or `open=False` would be more discoverable.
- **`vis_outpath` strips the extension** — the user must NOT include `.pdf`/`.png` in the path; TorchLens appends the extension from `vis_fileformat`. If you pass `my_path.pdf`, you get `my_path.pdf.pdf`. The error is silent (wrong file created). This is a common footgun.
- **`draw_backward()` needs `.backward()` called manually** — no error is raised if backward was not run; an empty or invalid graph is drawn silently. A `capture_backward=True` shortcut on `tl.trace` would be cleaner.
- **`show_containers=False` is the default but must be passed explicitly** — `False` (bool) vs `'labels'` (str) in the same kwarg is slightly awkward; could be unified under a single enum (`'none'|'labels'|'cluster'|...`).
- **No `return_graph` display path for notebooks** — `return_graph=True` returns the graphviz Source object; it does not render inline automatically. Users who want inline display must go through `vis_fileformat='png'` + `IPython.display.Image`. A Jupyter-native `display=True` shortcut would reduce boilerplate.